In [ ]:
!pip install transformers torch sentencepiece

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-large"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

In [ ]:
# essay = """
# The rapid development of artificial intelligence (AI) has revolutionized many industries. [MISSING]. However, ethical concerns about data privacy remain unresolved.
# """
# essay = "Neural networks demonstrate remarkable pattern recognition capabilities. [MISSING]. This makes them ideal for medical image analysis."

def gen_prompt(essay):
  prompt = f"""
  Task: Generate a coherent sentence to fill in the [MISSING] gap within an academic essay. The sentence must:
  1. Logically connect the preceding and following context
  2. Maintain formal academic tone
  3. Be 15-25 words long
  4. Include relevant keywords from the surrounding text
  Current Essay Context: {essay}
  Generated Sentence:
  """
  return prompt

In [ ]:
import pandas as pd
import nltk
import random
nltk.download('punkt_tab')

In [ ]:
df = pd.read_csv('ielts_dataset_v1.4_rawxparaphrased.csv')
df["replace_index"] = [None] * len(df)
df.head()

In [ ]:
def replacement_1(essay, mask_ratio=0.2):
  sentences = nltk.sent_tokenize(essay)
  num_to_mask = max(1, int(len(sentences) * mask_ratio))
  masked_indices = random.sample(range(len(sentences)), num_to_mask)
  for index in masked_indices:
    sentences[index] = "[MISSING]."
    input_essay = " ".join(sentences)
    new_sentence = gen_text(input_essay)
    # print(f"{index} : {new_sentence}")
    sentences[index] = new_sentence

  return masked_indices, " ".join(sentences)

In [ ]:
def replacement_2(essay, mask_ratio=0.2):
  sentences = nltk.sent_tokenize(essay)
  n = random.sample(range(5), 1)[0]
  masked_indices = [n + i for i in range(3)]

  for index in masked_indices:
    sentences[index] = "[MISSING]."
    input_essay = " ".join(sentences)
    new_sentence = gen_text(input_essay)
    # print(f"{index} : {new_sentence}")
    sentences[index] = new_sentence

  return masked_indices, " ".join(sentences)

In [ ]:
def gen_text(essay):
  input_text = gen_prompt(essay)
  inputs = tokenizer(
      input_text,
      return_tensors="pt",
      max_length=512,
      truncation=True,
      padding="max_length"
  )

  outputs = model.generate(
      input_ids=inputs.input_ids,
      attention_mask=inputs.attention_mask,
      do_sample=True,
      max_new_tokens=100,
      temperature=1.3,
      num_beams=5,
      early_stopping=True
  )

  result = tokenizer.decode(outputs[0], skip_special_tokens=True)
  return result

In [ ]:
# Main
for index, row in df.iterrows():
  if(row["is_ai"] == 1 and row["variant"] == 'raw'):
    essay = row["text"]
    a, b  = replacement_1(essay)
    df.at[index, "replace_index"] = a
    df.at[index, "text"] = b
  elif(row["is_ai"] == 0):
    essay = row["text"]
    a, b  = replacement_2(essay)
    df.at[index, "replace_index"] = a
    df.at[index, "text"] = b
    df.at[index, "variant"] = "word-sub-huai"
    df.at[index, "is_ai"] = 2
  else:
    continue

# df.head()
df.to_csv("sentences_substitution.csv", index=True)